In [1]:
import torch

print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능 여부: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 이름: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA를 사용할 수 없습니다.")

PyTorch 버전: 2.5.1+cu118
CUDA 사용 가능 여부: True
GPU 이름: NVIDIA GeForce RTX 4060 Ti


### 1. 시즌별로 정규시즌과 플레이오프 데이터 가져오기

In [2]:
from nba_api.stats.endpoints import leaguegamefinder
import pandas as pd
import time

def get_season_games(season, season_type):
    gamefinder = leaguegamefinder.LeagueGameFinder(
        season_nullable=season,
        season_type_nullable=season_type  # 'Regular Season' or 'Playoffs'
    )
    games = gamefinder.get_data_frames()[0]
    games['SEASON'] = season
    games['SEASON_TYPE'] = season_type
    return games

seasons = ['2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24']
all_games = []

for season in seasons:
    for stype in ['Regular Season', 'Playoffs']:
        print(f"Fetching {stype} for {season}")
        df = get_season_games(season, stype)
        all_games.append(df)
        time.sleep(1)

games_df = pd.concat(all_games, ignore_index=True)


Fetching Regular Season for 2017-18
Fetching Playoffs for 2017-18
Fetching Regular Season for 2018-19
Fetching Playoffs for 2018-19
Fetching Regular Season for 2019-20
Fetching Playoffs for 2019-20
Fetching Regular Season for 2020-21
Fetching Playoffs for 2020-21
Fetching Regular Season for 2021-22
Fetching Playoffs for 2021-22
Fetching Regular Season for 2022-23
Fetching Playoffs for 2022-23
Fetching Regular Season for 2023-24
Fetching Playoffs for 2023-24


### 2. 기본 전처리 (Team X Game, 홈/원정, 상대팀 등)

In [3]:
games_df['HOME'] = games_df['MATCHUP'].apply(lambda x: 1 if 'vs.' in x else 0)
games_df['OPPONENT'] = games_df['MATCHUP'].str.extract(r'vs\. (.+)|@ (.+)').bfill(axis=1).iloc[:, 0]
games_df['GAME_NUM'] = games_df.groupby(['SEASON', 'TEAM_NAME', 'SEASON_TYPE']).cumcount() + 1


### 3. 실점 추가 (상대팀과 merge)

In [4]:
# 득점 기준으로 1차 저장
pts_df = games_df[['GAME_ID', 'TEAM_NAME', 'PTS']]
pts_df.columns = ['GAME_ID', 'OPPONENT', 'OPP_PTS']

# merge하여 상대 득점 포함
games_df = games_df.merge(pts_df, on=['GAME_ID', 'OPPONENT'], how='left')


## 파생 변수
### 최근 5경기 평균 득점 / 실점 (팀 폼)

In [5]:
games_df = games_df.sort_values(['TEAM_NAME', 'GAME_DATE'])

games_df['AVG_PTS_5'] = games_df.groupby('TEAM_NAME')['PTS'].transform(lambda x: x.shift().rolling(5).mean())
games_df['AVG_OPP_PTS_5'] = games_df.groupby('TEAM_NAME')['OPP_PTS'].transform(lambda x: x.shift().rolling(5).mean())


### 피로도 지표: 백투백 여부

In [6]:
games_df['GAME_DATE'] = pd.to_datetime(games_df['GAME_DATE'])
games_df['PREV_GAME_DATE'] = games_df.groupby('TEAM_NAME')['GAME_DATE'].shift(1)
games_df['IS_BACK_TO_BACK'] = (games_df['GAME_DATE'] - games_df['PREV_GAME_DATE']).dt.days.eq(1).astype(int)


### 경기 순번 (파이널용 G1~G7)

In [7]:
games_df['GAME_NUM'] = games_df.groupby(['SEASON', 'TEAM_NAME', 'SEASON_TYPE']).cumcount() + 1


### 상대 전적

In [8]:
# 정규시즌만 필터
reg_df = games_df[games_df['SEASON_TYPE'] == 'Regular Season'].copy()

# 승/패 정보 추가
reg_df['WIN'] = (reg_df['WL'] == 'W').astype(int)
reg_df['LOSS'] = 1 - reg_df['WIN']

# 정렬
reg_df = reg_df.sort_values(['TEAM_NAME', 'OPPONENT', 'GAME_DATE'])

# 상대전적 누적 계산 (자기 팀 기준)
reg_df['CUM_WIN_VS_OPP'] = reg_df.groupby(['SEASON', 'TEAM_NAME', 'OPPONENT'])['WIN'].cumsum().shift().fillna(0)
reg_df['CUM_LOSS_VS_OPP'] = reg_df.groupby(['SEASON', 'TEAM_NAME', 'OPPONENT'])['LOSS'].cumsum().shift().fillna(0)

# 예시: 2023-24 BOS가 DEN과 경기 전까지 몇 승 몇 패였는지 확인 가능


### 승/패 (Target 변수) 생성

In [10]:
# 승/패 결과를 숫자로 변환 (1: 승리, 0: 패배)
games_df['WIN'] = (games_df['WL'] == 'W').astype(int)


### 우승 확률 (시즌 단위 모델 -> 경기 단위로 merge)

In [22]:
from nba_api.stats.endpoints import leaguedashteamstats
import time

team_stats_list = []

for season in seasons:  # seasons = ['2017-18', ..., '2023-24']
    print(f"Fetching team stats for {season}")
    try:
        stats = leaguedashteamstats.LeagueDashTeamStats(season=season).get_data_frames()[0]
        stats['SEASON'] = season
        team_stats_list.append(stats)
        time.sleep(0.7)
    except Exception as e:
        print(f"❌ Failed for {season}: {e}")


Fetching team stats for 2017-18
Fetching team stats for 2018-19
Fetching team stats for 2019-20
Fetching team stats for 2020-21
Fetching team stats for 2021-22
Fetching team stats for 2022-23
Fetching team stats for 2023-24


In [23]:
team_stats_df = pd.concat(team_stats_list, ignore_index=True)


In [26]:
season_stats = team_stats_df[[
    'SEASON', 'TEAM_NAME',
    'W_PCT',        # 승률
    'PTS',          # 평균 득점
    'FG_PCT',       # 야투율
    'FG3_PCT',      # 3점슛율
    'FT_PCT',       # 자유투율
    'REB',          # 리바운드
    'AST',          # 어시스트
    'TOV',          # 턴오버
    'STL',          # 스틸
    'BLK',          # 블락
    'PLUS_MINUS'    # 팀 +/- 지표
]].copy()


In [27]:
champions = {
    '2017-18': 'Golden State Warriors',
    '2018-19': 'Toronto Raptors',
    '2019-20': 'Los Angeles Lakers',
    '2020-21': 'Milwaukee Bucks',
    '2021-22': 'Golden State Warriors',
    '2022-23': 'Denver Nuggets',
    '2023-24': 'Boston Celtics'
}

season_stats['IS_CHAMPION'] = season_stats.apply(
    lambda row: int(row['TEAM_NAME'] == champions.get(row['SEASON'], '')),
    axis=1
)


In [28]:
from xgboost import XGBClassifier

X = season_stats.drop(columns=['SEASON', 'TEAM_NAME', 'IS_CHAMPION'])
y = season_stats['IS_CHAMPION']

model = XGBClassifier()
model.fit(X, y)

# 예측 확률 생성
season_stats['WIN_PROB'] = model.predict_proba(X)[:, 1]


In [29]:
games_df = games_df.merge(
    season_stats[['SEASON', 'TEAM_NAME', 'WIN_PROB']],
    on=['SEASON', 'TEAM_NAME'],
    how='left'
)


### 부상 선수 수

In [13]:
# 로스터 대시보드에서 경기별 출전 선수 수 조회 가능 (단, 매 경기 추출은 매우 느림)
# 여기선 평균적으로 7명 이하 출전 시 부상 영향 있다고 가정

games_df['LOW_PLAYER_COUNT'] = (games_df['MIN'] < 7 * 12).astype(int)


### 시즌 평균 스탯 병합

In [36]:
from nba_api.stats.endpoints import leaguedashteamstats
import time
import pandas as pd

# ✅ 1. 시즌 리스트 정의
seasons = ['2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24']

# ✅ 2. 시즌별 팀 통계 수집
team_stats = []
for season in seasons:
    print(f"📥 Fetching season stats for {season}")
    stats = leaguedashteamstats.LeagueDashTeamStats(season=season).get_data_frames()[0]
    stats['SEASON'] = season
    team_stats.append(stats)
    time.sleep(1)

# ✅ 3. 통합 DataFrame 생성
team_stats_df = pd.concat(team_stats, ignore_index=True)

selected_columns = [
    'TEAM_NAME', 'SEASON',
    'PTS',          # 평균 득점
    'FG_PCT',       # 야투율
    'FG3_PCT',      # 3점슛율
    'FT_PCT',       # 자유투율
    'AST',          # 어시스트 수
    'REB',          # 리바운드 수
    'STL', 'BLK',   # 수비 지표
    'TOV',          # 턴오버
    'PLUS_MINUS',   # +/- 지표
    'W_PCT'         # 시즌 승률
]

# 병합
games_df = games_df.merge(
    team_stats_df[selected_columns],
    on=['TEAM_NAME', 'SEASON'],
    how='left'
)

# 저장
games_df.to_csv("C:/Users/THKIM/Desktop/3-1 프로젝트/오픈데이터분석 프로젝트/우승팀 예측/데이터 셋/games_dataset_final.csv", index=False, encoding='utf-8-sig')

print("✅ 저장 완료: games_dataset_final.csv")



📥 Fetching season stats for 2017-18
📥 Fetching season stats for 2018-19
📥 Fetching season stats for 2019-20
📥 Fetching season stats for 2020-21
📥 Fetching season stats for 2021-22
📥 Fetching season stats for 2022-23
📥 Fetching season stats for 2023-24
✅ 저장 완료: games_dataset_final.csv
